<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/StableDiffusionChapter10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    "stablediffusionapi/deliberate-v2"
    , torch_dtype=torch.float16
).to("cuda")

prompt = "a photo of a cat and a dog driving an aircraft "*20
image = pipe(
    prompt = prompt
).images[0]
image

In [ ]:
prompt = "a photo of a cat and a dog driving an aircraft "*20

In [ ]:
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

In [ ]:
tokens = tokenizer(
    prompt
    , truncation = False
    , return_tensors = 'pt'
)["input_ids"]
print(len(tokens[0]))

In [ ]:
embeddings = pipe.text_encoder(
    tokens.to("cuda")
)[0]

In [ ]:
prompt = "a photo of a cat and a dog driving an aircraft "

In [ ]:
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

In [ ]:
tokens = tokenizer(
    prompt
    , truncation = False
    , return_tensors = 'pt'
)["input_ids"]
print(len(tokens[0]))

In [ ]:
embeddings = pipe.text_encoder(
    tokens.to("cuda")
)[0]

In [ ]:
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder

In [ ]:
tokens = tokenizer(
    prompt
    , truncation = False
)["input_ids"]
print("token length:", len(tokens))

In [ ]:
from transformers import CLIPTokenizer
neg_prompt = "a bad photo of a cat"

In [ ]:
negative_ids = pipe.tokenizer(
    neg_prompt
    , truncation    = False
    , padding       = "max_length"
    , max_length    = len(tokens)
).input_ids
print("neg_token length:", len(negative_ids))

In [ ]:
tokens = tokens[1:-1]
negative_ids = negative_ids[1:-1]

In [ ]:
embeds,neg_embeds = [],[]
chunk_size = 75
bos = pipe.tokenizer.bos_token_id
eos = pipe.tokenizer.eos_token_id
for i in range(0, len(tokens), chunk_size):
    # Add begin and end token to the 75 chunked tokens to make a 77 token list
    sub_tokens          = [bos] + tokens[i:i + chunk_size] + [eos]

    # text_encoder support torch.Size([1,x]) input tensor
    # that is why use [sub_tokens], instead of simply give sub_tokens.
    tensor_tokens       = torch.tensor(
        [sub_tokens]
        , dtype = torch.long
        , device = pipe.device
    )
    chunk_embeds        = text_encoder(tensor_tokens)[0]
    embeds.append(chunk_embeds)

    # Add begin and end token to the 75 chunked neg tokens to make a 77 token list
    sub_neg_tokens      = [bos] + negative_ids[i:i + chunk_size] + [eos]
    tensor_neg_tokens   = torch.tensor(
        [sub_neg_tokens]
        , dtype = torch.long
        , device = pipe.device
    )
    neg_chunk_embeds    = text_encoder(tensor_neg_tokens)[0]
    neg_embeds.append(neg_chunk_embeds)

In [ ]:
prompt_embeds = torch.cat(embeds, dim = 1)
prompt_neg_embeds = torch.cat(neg_embeds, dim = 1)

In [ ]:
def long_prompt_encoding(
    pipe:StableDiffusionPipeline
    , prompt
    , neg_prompt = ""
):
    bos = pipe.tokenizer.bos_token_id
    eos = pipe.tokenizer.eos_token_id
    chunk_size = 75

    # step 1. take out the tokenizer and text encoder
    tokenizer = pipe.tokenizer
    text_encoder = pipe.text_encoder

    # step 2.1. encode whatever size prompt to tokens by setting truncation = False.
    tokens = tokenizer(
        prompt
        , truncation = False
        #, return_tensors = 'pt'
    )["input_ids"]

    # step 2.2. encode whatever size neg_prompt, padding it to the size of prompt.
    negative_ids = pipe.tokenizer(
        neg_prompt
        , truncation    = False
        #, return_tensors = "pt"
        , padding       = "max_length"
        , max_length    = len(tokens)
    ).input_ids

    # Step 3. remove begin and end tokens
    tokens = tokens[1:-1]
    negative_ids = negative_ids[1:-1]

    # step 4. Pop out the head 77 tokens, and encode the 77 tokens to embeddings.
    embeds,neg_embeds = [],[]
    for i in range(0, len(tokens), chunk_size):
        # Add begin and end token to the 75 chunked tokens to make a 77 token list
        sub_tokens          = [bos] + tokens[i:i + chunk_size] + [eos]

        # text_encoder support torch.Size([1,x]) input tensor
        # that is why use [sub_tokens], instead of simply give sub_tokens.
        tensor_tokens       = torch.tensor(
            [sub_tokens]
            , dtype = torch.long
            , device = pipe.device
        )
        chunk_embeds        = text_encoder(tensor_tokens)[0]
        embeds.append(chunk_embeds)

        # Add begin and end token to the 75 chunked neg tokens to make a 77 token list
        sub_neg_tokens      = [bos] + negative_ids[i:i + chunk_size] + [eos]
        tensor_neg_tokens   = torch.tensor(
            [sub_neg_tokens]
            , dtype = torch.long
            , device = pipe.device
        )
        neg_chunk_embeds    = text_encoder(tensor_neg_tokens)[0]
        neg_embeds.append(neg_chunk_embeds)

    # step 5. Stack the embeddings to a [1,x,768] size torch tensor.
    prompt_embeds = torch.cat(embeds, dim = 1)
    prompt_neg_embeds = torch.cat(neg_embeds, dim = 1)

    return prompt_embeds, prompt_neg_embeds

In [ ]:
prompt = "photo, cute cat running on the grass" * 10 #<- long prompt
prompt_embeds, prompt_neg_embeds = long_prompt_encoding(
    pipe, prompt, neg_prompt="low resolution, bad anatomy"
)
print(prompt_embeds.shape)

image = pipe(
    prompt_embeds = prompt_embeds
    , negative_prompt_embeds = prompt_neg_embeds
    , generator = torch.Generator("cuda").manual_seed(1)
).images[0]
image

In [ ]:
prompt = "photo, cute cat running on the grass" * 10
prompt = prompt + ",pure white cat" * 10

In [ ]:
def parse_prompt_attention(text):
    import re
    re_attention = re.compile(
        r"""
            \\\(|\\\)|\\\[|\\]|\\\\|\\|\(|\[|:([+-]?[.\d]+)\)|
            \)|]|[^\\()\[\]:]+|:
        """
        , re.X
    )

    re_break = re.compile(r"\s*\bBREAK\b\s*", re.S)

    res = []
    round_brackets = []
    square_brackets = []

    round_bracket_multiplier = 1.1
    square_bracket_multiplier = 1 / 1.1

    def multiply_range(start_position, multiplier):
        for p in range(start_position, len(res)):
            res[p][1] *= multiplier

    for m in re_attention.finditer(text):
        text = m.group(0)
        weight = m.group(1)

        if text.startswith('\\'):
            res.append([text[1:], 1.0])
        elif text == '(':
            round_brackets.append(len(res))
        elif text == '[':
            square_brackets.append(len(res))
        elif weight is not None and len(round_brackets) > 0:
            multiply_range(round_brackets.pop(), float(weight))
        elif text == ')' and len(round_brackets) > 0:
            multiply_range(round_brackets.pop(), round_bracket_multiplier)
        elif text == ']' and len(square_brackets) > 0:
            multiply_range(square_brackets.pop(), square_bracket_multiplier)
        else:
            parts = re.split(re_break, text)
            for i, part in enumerate(parts):
                if i > 0:
                    res.append(["BREAK", -1])
                res.append([part, 1.0])

    for pos in round_brackets:
        multiply_range(pos, round_bracket_multiplier)

    for pos in square_brackets:
        multiply_range(pos, square_bracket_multiplier)

    if len(res) == 0:
        res = [["", 1.0]]

    # merge runs of identical weights
    i = 0
    while i + 1 < len(res):
        if res[i][1] == res[i + 1][1]:
            res[i][0] += res[i + 1][0]
            res.pop(i + 1)
        else:
            i += 1
    return res

In [ ]:
parse_prompt_attention("a (white) cat")

In [ ]:
def get_prompts_tokens_with_weights(
    pipe: StableDiffusionPipeline
    , prompt: str
):
    texts_and_weights = parse_prompt_attention(prompt)
    text_tokens,text_weights = [],[]
    for word, weight in texts_and_weights:
        # tokenize and discard the starting and the ending token
        token = pipe.tokenizer(
            word
            , truncation = False # so that tokenize whatever length prompt
        ).input_ids[1:-1]
        # the returned token is a 1d list: [320, 1125, 539, 320]

        # use merge the new tokens to the all tokens holder: text_tokens
        text_tokens = [*text_tokens,*token]

        # each token chunk will come with one weight, like ['red cat', 2.0]
        # need to expand weight for each token.
        chunk_weights = [weight] * len(token)

        # append the weight back to the weight holder: text_weights
        text_weights = [*text_weights, *chunk_weights]
    return text_tokens,text_weights

In [ ]:
prompt = "a (white) cat"
tokens, weights = get_prompts_tokens_with_weights(pipe, prompt)
print(tokens,weights)

In [ ]:
# encode "white" only
white_token = 1579
white_token_tensor = torch.tensor(
    [[white_token]]
    , dtype = torch.long
    , device = pipe.device
)
white_embed = pipe.text_encoder(white_token_tensor)[0]
print(white_embed[0][0])

In [ ]:
# encode "white cat"
white_token, cat_token = 1579, 2369
white_cat_token_tensor = torch.tensor(
    [[white_token, cat_token]]
    , dtype = torch.long
    , device = pipe.device
)
white_cat_embeds = pipe.text_encoder(white_cat_token_tensor)[0]
print(white_cat_embeds[0][0])

In [ ]:
# step 3. padding tokens
def pad_tokens_and_weights(
    token_ids: list
    , weights: list
):
    bos,eos = 49406,49407

    # this will be a 2d list
    new_token_ids = []
    new_weights   = []
    while len(token_ids) >= 75:
        # get the first 75 tokens
        head_75_tokens = [token_ids.pop(0) for _ in range(75)]
        head_75_weights = [weights.pop(0) for _ in range(75)]

        # extract token ids and weights
        temp_77_token_ids = [bos] + head_75_tokens + [eos]
        temp_77_weights   = [1.0] + head_75_weights + [1.0]

        # add 77 token and weights chunk to the holder list
        new_token_ids.append(temp_77_token_ids)
        new_weights.append(temp_77_weights)

    # padding the left
    if len(token_ids) > 0:
        padding_len         = 75 - len(token_ids)
        padding_len = 0

        temp_77_token_ids   = [bos] + token_ids + [eos] * padding_len + [eos]
        new_token_ids.append(temp_77_token_ids)

        temp_77_weights     = [1.0] + weights   + [1.0] * padding_len + [1.0]
        new_weights.append(temp_77_weights)

    # return
    return new_token_ids, new_weights

In [ ]:
t,w = pad_tokens_and_weights(tokens.copy(), weights.copy())
print(t)
print(w)

In [ ]:
def get_weighted_text_embeddings(
    pipe: StableDiffusionPipeline
    , prompt : str      = ""
    , neg_prompt: str   = ""
):
    eos = pipe.tokenizer.eos_token_id
    prompt_tokens, prompt_weights = get_prompts_tokens_with_weights(
        pipe, prompt
    )
    neg_prompt_tokens, neg_prompt_weights = get_prompts_tokens_with_weights(
        pipe, neg_prompt
    )

    # padding the shorter one
    prompt_token_len        = len(prompt_tokens)
    neg_prompt_token_len    = len(neg_prompt_tokens)
    if prompt_token_len > neg_prompt_token_len:
        # padding the neg_prompt with eos token
        neg_prompt_tokens   = (
            neg_prompt_tokens  +
            [eos] * abs(prompt_token_len - neg_prompt_token_len)
        )
        neg_prompt_weights  = (
            neg_prompt_weights +
            [1.0] * abs(prompt_token_len - neg_prompt_token_len)
        )
    else:
        # padding the prompt
        prompt_tokens       = (
            prompt_tokens
            + [eos] * abs(prompt_token_len - neg_prompt_token_len)
        )
        prompt_weights      = (
            prompt_weights
            + [1.0] * abs(prompt_token_len - neg_prompt_token_len)
        )

    embeds = []
    neg_embeds = []

    prompt_token_groups ,prompt_weight_groups = pad_tokens_and_weights(
        prompt_tokens.copy()
        , prompt_weights.copy()
    )

    neg_prompt_token_groups, neg_prompt_weight_groups = pad_tokens_and_weights(
        neg_prompt_tokens.copy()
        , neg_prompt_weights.copy()
    )

    # get prompt embeddings one by one is not working.
    for i in range(len(prompt_token_groups)):
        # get positive prompt embeddings with weights
        token_tensor = torch.tensor(
            [prompt_token_groups[i]]
            ,dtype = torch.long, device = pipe.device
        )
        weight_tensor = torch.tensor(
            prompt_weight_groups[i]
            , dtype     = torch.float16
            , device    = pipe.device
        )
        token_embedding = pipe.text_encoder(token_tensor)[0].squeeze(0)
        for j in range(len(weight_tensor)):
            token_embedding[j] = token_embedding[j] * weight_tensor[j]
        token_embedding = token_embedding.unsqueeze(0)
        embeds.append(token_embedding)

        # get negative prompt embeddings with weights
        neg_token_tensor = torch.tensor(
            [neg_prompt_token_groups[i]]
            , dtype = torch.long, device = pipe.device
        )
        neg_weight_tensor = torch.tensor(
            neg_prompt_weight_groups[i]
            , dtype     = torch.float16
            , device    = pipe.device
        )
        neg_token_embedding = pipe.text_encoder(neg_token_tensor)[0].squeeze(0)
        for z in range(len(neg_weight_tensor)):
            neg_token_embedding[z] = (
                neg_token_embedding[z] * neg_weight_tensor[z]
            )
        neg_token_embedding = neg_token_embedding.unsqueeze(0)
        neg_embeds.append(neg_token_embedding)

    prompt_embeds       = torch.cat(embeds, dim = 1)
    neg_prompt_embeds   = torch.cat(neg_embeds, dim = 1)

    return prompt_embeds, neg_prompt_embeds

In [ ]:
prompt = "photo, cute cat running on the grass" * 10
prompt = prompt + ",pure (white:1.5) cat" * 10

neg_prompt = "low resolution, bad anatomy"

prompt_embeds, prompt_neg_embeds = get_weighted_text_embeddings(pipe, prompt = prompt, neg_prompt = neg_prompt)

image = pipe(
    prompt_embeds = prompt_embeds
    , negative_prompt_embeds = prompt_neg_embeds
    , generator = torch.Generator("cuda").manual_seed(1)
).images[0]
image

In [ ]:
from diffusers import DiffusionPipeline
import torch

model_id_or_path = "stablediffusionapi/deliberate-v2"
pipe = DiffusionPipeline.from_pretrained(
    model_id_or_path
    , torch_dtype       = torch.float16
    , custom_pipeline   = "lpw_stable_diffusion"
).to("cuda:0")

In [ ]:
prompt = "photo, cute cat running on the grass" * 10
prompt = prompt + ",pure (white:1.5) cat" * 10

neg_prompt = "low resolution, bad anatomy"
image = pipe(
    prompt = prompt
    , negative_prompt = neg_prompt
    , generator = torch.Generator("cuda").manual_seed(1)
).images[0]
image

In [ ]:
!pip install accelerate

In [ ]:
from diffusers import DiffusionPipeline
import torch

model_id_or_path = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = DiffusionPipeline.from_pretrained(
    model_id_or_path
    , torch_dtype       = torch.float16
    , custom_pipeline   = "lpw_stable_diffusion_xl",
).to("cuda:0")

In [ ]:
prompt = "photo, cute cat running on the grass" * 10
prompt = prompt + ",pure (white:1.5) cat" * 10

neg_prompt = "low resolution, bad anatomy"
image = pipe(
    prompt = prompt
    , negative_prompt = neg_prompt
    , generator = torch.Generator("cuda").manual_seed(7)
).images[0]
image